In [2]:
%pip install libpysal

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install --force-reinstall "pandas<2.0.0"

  Using cached pandas-1.5.3.tar.gz (5.2 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [20 lines of output]
      Traceback (most recent call last):
        File "c:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\env\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 353, in <module>
          main()
        File "c:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\env\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\env\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 118, in get_requires_for_build_wheel
          return hook(config_settings)
                 ^^^^^^^^^

In [4]:
from SEDataObjects import *
from SEDataObjects.BaseLayer import *
from SEDataObjects.transitWrappers import *
from SEDataObjects.constants import GEODESIC_CRS
import geopandas as gpd
import folium

In [5]:
CONFIG_MAP_AREA_PATH = "map_area_path"
CONFIG_MAP_AREA_PROJECTED_CRS = "map_area_projected_crs"
CONFIG_GTFS_TRANSITLAND_URL = "gtfs_url"
CONFIG_GTFS_TRANSITLAND_KEY_PATH = "gtfs_key_path"
CONFIG_GTFS_CACHE_FOLDER = "gtfs_cache_folder"
CONFIG_FTA_FACILITY_INVENTORY_PATH = "fta_facility_inventory_path"
CONFIG_CITYBIKES_URL = "citybikes_url"
CONFIG_AFDC_API_KEY = "afdc_api_key"
CONFIG_AFDC_URL = "afdc_url"
CONFIG_OSM_CACHE_FOLDER = "osm_cache_folder"
CONFIG_EPA_EJSCREEN_PATH = "epa_ejscreen_path"
CONFIG_SMART_LOCATION_PATH = "smart_location_path"

# COMPLETE CONFIG HERE
CONFIG = {
    CONFIG_MAP_AREA_PATH: "./rawData/san-francisco-2010.geojson", # The path to the map area spatial data file
    CONFIG_MAP_AREA_PROJECTED_CRS: 3310, # The EPSG number for a projected crs with units in meters covering the map area
    CONFIG_GTFS_TRANSITLAND_URL: "https://transit.land/api/v2/rest/feeds.json", # A path to the Transitland V2 api. See https://www.interline.io/transitland/plans-pricing/
    CONFIG_GTFS_TRANSITLAND_KEY_PATH: "./rawData/TRANSITLAND_KEY.txt", # A path to a file contaiing your Transitland API key
    CONFIG_GTFS_CACHE_FOLDER: "./cache/gtfs_cache/SF_county", # A path to a folder where GTFS-static feeds and extracted data canbe stored. The folder must exist, and is recommended to be in cache/gtfs_cache/**
    CONFIG_FTA_FACILITY_INVENTORY_PATH: "./rawData/2023 Facility Inventory.xlsx", # A path to the FTA facility inventory sheet
    CONFIG_CITYBIKES_URL: "http://api.citybik.es/", # The Citybikes API endpoint. See https://citybik.es
    CONFIG_OSM_CACHE_FOLDER: "./cache/osmnx_cache", # A folder to use as the OSMNX cache
    CONFIG_AFDC_URL: "https://developer.nlr.gov/api/alt-fuel-stations/v1/nearest.geojson", # A url to the NRL AFDC alternate fuel stations nearby api endpoint
    CONFIG_AFDC_API_KEY: "./rawData/AFDC_API_KEY.txt", # A path to a folder with a NREL api key. See https://developer.nrel.gov/docs/api-key/
    CONFIG_EPA_EJSCREEN_PATH: "rawData/EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb", # A path to the Ejscreen gdb
    CONFIG_SMART_LOCATION_PATH: "rawData/SmartLocationDatabaseV3/SmartLocationDatabase.gdb", # A path to the Smart Location DB gdb
}

In [6]:
import pandas as pd

# Load your uploaded CSV
carless_df = pd.read_csv("./rawData/SF_car_ownership_2019.csv")

# Print columns and the first 2 rows to see the structure
print("Columns in CSV:", carless_df.columns.tolist())
display(carless_df.head(2))

Columns in CSV: ['GEOID', 'Geographic Area Name', 'Estimate!!Total:', 'Estimate!!Total:!!Owner occupied:', 'Estimate!!Total:!!Owner occupied:!!No vehicle available', 'Estimate!!Total:!!Renter occupied:', 'Estimate!!Total:!!Renter occupied:!!No vehicle available', 'carlessness total count', 'carlessness ratio', 'Margin of Error!!Total:!!Renter occupied:!!No vehicle available', 'Estimate!!Total:!!Renter occupied:!!1 vehicle available', 'Margin of Error!!Total:!!Renter occupied:!!1 vehicle available', 'Estimate!!Total:!!Renter occupied:!!2 vehicles available', 'Margin of Error!!Total:!!Renter occupied:!!2 vehicles available', 'Estimate!!Total:!!Renter occupied:!!3 vehicles available', 'Margin of Error!!Total:!!Renter occupied:!!3 vehicles available', 'Estimate!!Total:!!Renter occupied:!!4 vehicles available', 'Margin of Error!!Total:!!Renter occupied:!!4 vehicles available', 'Estimate!!Total:!!Renter occupied:!!5 or more vehicles available', 'Margin of Error!!Total:!!Renter occupied:!!5 o

,GEOID,Geographic Area Name,Estimate!!Total:,Estimate!!Total:!!Owner occupied:,Estimate!!Total:!!Owner occupied:!!No vehicle available,Estimate!!Total:!!Renter occupied:,Estimate!!Total:!!Renter occupied:!!No vehicle available,carlessness total count,carlessness ratio,Margin of Error!!Total:!!Renter occupied:!!No vehicle available,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,1500000US060750101001,"Block Group 1, Census Tract 101, San Francisco...",484,0,0,0,0,0,0.000000,12,...,222,98,9,15,0,12,0,12,0,12
1,1500000US060750101002,"Block Group 2, Census Tract 101, San Francisco...",1544,171,14,122,20,34,0.022021,25,...,525,119,144,111,0,12,0,12,0,12


In [ ]:
carless_df['tract_key'] = (
    carless_df['GEOID']
    .astype(str)
    .str.replace('1500000US', '', regex=False) # Remove the metadata prefix
    .str[:11]                                  # Keep exactly the 11-digit tract ID
)

,GEOID,Geographic Area Name,Estimate!!Total:,Estimate!!Total:!!Owner occupied:,Estimate!!Total:!!Owner occupied:!!No vehicle available,Estimate!!Total:!!Renter occupied:,Estimate!!Total:!!Renter occupied:!!No vehicle available,carlessness total count,carlessness ratio,Margin of Error!!Total:!!Renter occupied:!!No vehicle available,...,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,tract_key
0,1500000US060750101001,"Block Group 1, Census Tract 101, San Francisco...",484,0,0,0,0,0,0.000000,12,...,98,9,15,0,12,0,12,0,12,06075010100
1,1500000US060750101002,"Block Group 2, Census Tract 101, San Francisco...",1544,171,14,122,20,34,0.022021,25,...,119,144,111,0,12,0,12,0,12,06075010100
2,1500000US060750102001,"Block Group 1, Census Tract 102, San Francisco...",862,339,48,192,99,147,0.170534,62,...,92,67,46,0,12,0,12,0,12,06075010200
3,1500000US060750102002,"Block Group 2, Census Tract 102, San Francisco...",1059,320,36,184,78,114,0.107649,59,...,112,55,52,0,12,0,12,18,28,06075010200
4,1500000US060750102003,"Block Group 3, Census Tract 102, San Francisco...",594,308,46,172,90,136,0.228956,59,...,71,65,55,0,12,0,12,0,12,06075010200


In [14]:
# Define map area
map_area_gdf = gpd.read_file(CONFIG[CONFIG_MAP_AREA_PATH]).to_crs(GEODESIC_CRS)

centroids = map_area_gdf.geometry.centroid
map_area_gdf = map_area_gdf[
    (centroids.x > -122.55) & 
    (centroids.y < 37.83)
].copy()

# 3. Dissolve the remaining main peninsula shapes for the loaders
map_area = map_area_gdf.unary_union

C:\Users\caleb\AppData\Local\Temp\ipykernel_16660\587369639.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = map_area_gdf.geometry.centroid
C:\Users\caleb\AppData\Local\Temp\ipykernel_16660\587369639.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  map_area = map_area_gdf.unary_union


In [15]:
# Instantiate data objects

gtfs_instance = GTFSDataObject(
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    CONFIG[CONFIG_GTFS_CACHE_FOLDER],
    CONFIG[CONFIG_GTFS_TRANSITLAND_URL],
    api_key_path=CONFIG[CONFIG_GTFS_TRANSITLAND_KEY_PATH],
    download_transitland_first=True,  # <-- ADD THIS: Stops it from hitting the web
    load_from_cache=False              # <-- ADD THIS: Forces it to read local files
)

In [16]:
import pandas as pd

# Overwrite df_feeds_metadata with a customized version that handles dictionary assignment smoothly
original_init = pd.DataFrame.__init__

def patched_init(self, *args, **kwargs):
    original_init(self, *args, **kwargs)
    # Give the DataFrame's .loc a fallback mapping capability for this specific notebook run
    if hasattr(self, 'columns') and 'last_fetch_succeeded' in args or 'columns' in kwargs:
        pass

# Force the instance to ignore the dictionary strictness by patching the specific object instance
if hasattr(gtfs_instance, 'load_data'):
    # Let's bypass the strict dataframe construction line entirely by preemptively creating the dataframe structure it wants
    import datetime as dt
    columns_needed = [
        "name", "agency_url", "url", "raw_feed_path", "last_fetched", 
        "attribution_url", "attribution_text", "attribution_instructions", 
        "attribution_must_attribute", "last_fetch_succeeded"
    ]
    # Inject a cleanly built dataframe directly into the object so it doesn't try to build a broken empty one on line 159
    gtfs_instance.df_feeds_metadata = pd.DataFrame(columns=columns_needed)

In [17]:
import numpy as np
fta_instance = FTAFacilityInventoryDataObject(
    CONFIG[CONFIG_FTA_FACILITY_INVENTORY_PATH],
    CONFIG[CONFIG_OSM_CACHE_FOLDER]
)
citybikes_instance = CityBikesDataObject(CONFIG[CONFIG_CITYBIKES_URL])
afdc_instance = AFDCApiDataObject(CONFIG[CONFIG_AFDC_URL], CONFIG[CONFIG_AFDC_API_KEY],CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
osm_bike_parking_instance = OSMBikeParkingDataObject(CONFIG[CONFIG_OSM_CACHE_FOLDER], {"amenity": ["bicycle_parking"]})
smart_location_info = SmartLocationWrapper(CONFIG[CONFIG_SMART_LOCATION_PATH], CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])

base_layer_instance = BaseLayer(
    [
        SmartLocationPopulationDensity(smart_location_info),
        SmartLocationJobDensity(smart_location_info),
        SmartLocationRetailEntertainmentJobDensity(smart_location_info),
        SmartLocationRawJobs(smart_location_info),
        #CensusModeshare(),
        #CensusCarlessness(),
        SmartLocationNationalWalkabilityIndex(smart_location_info),
        BaseLayerEjscreen(CONFIG[CONFIG_EPA_EJSCREEN_PATH]),
    ],
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    ColorMaps.NATIONAL_WALKABILITY_INDEX_COLORMAP,
    smooth=True,
    remove_water=True # This should be set to False if TIGER goes down
)
bike_instance = OSMBikeStreetsDataObject(
    CONFIG[CONFIG_OSM_CACHE_FOLDER],
    reference=gtfs_instance,
    local_crs=CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS]
)

# Edit these columns to change the objects displayed on the map
all_objects = {
    "GTFS": gtfs_instance,
    "BASE": base_layer_instance,
    "BIKE": bike_instance,
    #"FTA": fta_instance,
    "CITYBIKES": citybikes_instance,
    "AFDC": afdc_instance,
    "OSM BIKE PARKING": osm_bike_parking_instance,
    }

objects_load_order = (
    "BASE",
    "BIKE",
    "GTFS",
    #"FTA",
    "AFDC",
    "OSM BIKE PARKING",
    "CITYBIKES"
)

# This should only contain GTFS
objects_must_await = (
    "GTFS"
)
await gtfs_instance.load_data(map_area, GEODESIC_CRS)
base_layer_instance.load_data(map_area, GEODESIC_CRS)

INFO: Transitland URL: https://transit.land/api/v2/rest/feeds.json?bbox=-122.534088,37.7080849990001,-122.28178,37.9298239990001&limit=100&license_create_derived_product=exclude_no&license_redistribution_allowed=exclude_no&apikey=Fk5jQ7yU3syJZF9a2A2VVjPq8FZyl6GN
INFO: Processing f-9-amtrak~amtrakcalifornia~amtrakcharteredvehicle
WARN: For f-9-amtrak~amtrakcalifornia~amtrakcharteredvehicle, the hash cef4d7e19a14ec235c32f74107b23a55d3236cad does not match the provided hash from Transitland 14f3753086a8251ff2199ade2caf40c650dd0023
Loading f-9-amtrak~amtrakcalifornia~amtrakcharteredvehicle
DEBUG: Attempting to open file path: C:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\cache\gtfs_cache\SF_county\gtfs_f-9-amtrak~amtrakcalifornia~amtrakcharteredvehicle.zip
partridge loaded
loading stop tuple
loaded stop tuple
trip patterns loaded
feed loaded
FEED NAME: Agency has Multiple Names
Feed f-9-amtrak~amtrakcalifornia~amtrakcharteredvehicle added
INFO: Processing f

c:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\env\Lib\site-packages\numpy\lib\_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in parse_time (vectorized)
  outputs = ufunc(*args, out=...)


loading stop tuple
loaded stop tuple
trip patterns loaded
feed loaded
FEED NAME: Agency has Multiple Names
Feed f-sf~bay~area~rg added
INFO: Processing f-stan~rta~modesto~newman~oakdale~pattersomand~riverbank~flex
Loading f-stan~rta~modesto~newman~oakdale~pattersomand~riverbank~flex
DEBUG: Attempting to open file path: C:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\cache\gtfs_cache\SF_county\gtfs_f-stan~rta~modesto~newman~oakdale~pattersomand~riverbank~flex.zip
partridge loaded


c:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\env\Lib\site-packages\numpy\lib\_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in parse_time (vectorized)
  outputs = ufunc(*args, out=...)


INFO: Processing f-west~cat~senior~dial~a~rideand~paratransit~flex
WARN: File downloaded, but a zip file was not returned
Loading f-west~cat~senior~dial~a~rideand~paratransit~flex
DEBUG: Attempting to open file path: C:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\cache\gtfs_cache\SF_county\gtfs_f-west~cat~senior~dial~a~rideand~paratransit~flex.zip
partridge loaded
loading stop tuple
loaded stop tuple
trip patterns loaded
feed loaded


...

FEED NAME: WestCat (Western Contra Costa)
Feed f-west~cat~senior~dial~a~rideand~paratransit~flex added
INFO: Processing f-lawrence~berkeley~national~lab
WARN: For f-lawrence~berkeley~national~lab, the hash 46cdd20d4830e1150ca794e8108decabe8ecd1ef does not match the provided hash from Transitland 025398d8f03eec84ca9f17c257e097677b9ed852
Loading f-lawrence~berkeley~national~lab
DEBUG: Attempting to open file path: C:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\cache\gtfs_cache\SF_county\gtfs_f-lawrence~berkeley~national~lab.zip
partridge loaded
INFO: Processing f-kaiser~permanente~east~bay~shuttle
WARN: For f-kaiser~permanente~east~bay~shuttle, the hash 3f0cab1f821b23ce3304971dcb9bde235c47c94e does not match the provided hash from Transitland 3736eb2822fa24adea0b606b751d8eb7e2bf20ab
Loading f-kaiser~permanente~east~bay~shuttle
DEBUG: Attempting to open file path: C:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\cache\gtfs_cach

c:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\env\Lib\site-packages\numpy\lib\_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in parse_time (vectorized)
  outputs = ufunc(*args, out=...)


Loading f-9q9-sanjoaquins
DEBUG: Attempting to open file path: C:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\cache\gtfs_cache\SF_county\gtfs_f-9q9-sanjoaquins.zip
partridge loaded
loading stop tuple
loaded stop tuple
trip patterns loaded
feed loaded
FEED NAME: Gold Runner
Feed f-9q9-sanjoaquins added


c:\Users\caleb\Desktop\SUMC\Task B\se_analysis_tool-main\se_analysis_tool-main\SEDataObjects\transitWrappers\TransitNetwork.py:939: PerformanceWarning: indexing past lexsort depth may impact performance.
  get_as_tuple(service_patterns_by_next_current_stop.loc[row["stop_id_unique"], row["next_stop"]])


Loaded Population Density (people/acre)
Loaded Job Density (jobs/acre)
Loaded Retail and Entertainment Job Density (jobs/acre)
Loaded Total # Jobs
Loaded National Walkability Index
Loaded National Traffic Proximity & Volume Percentile


#### Mobility Hub Map

In [36]:
# Mobility Hub Only Map
mobility_hub_instance = MobilityHubDataObject(gtfs_instance, base_layer_instance, CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
mobility_hub_instance.load_data(map_area, GEODESIC_CRS)
mobility_hub_map_old = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="CartoDB Voyager",
    zoom_start=12,
    prefer_canvas=True
)
base_layer_instance.get_folium_plot().add_to(mobility_hub_map_old)
mobility_hub_instance.get_folium_plot().add_to(mobility_hub_map_old)
mobility_hub_map_old.save("mobility_hub_map_old.html")

In [29]:
import folium

mobility_hub_instance = MobilityHubDataObject(gtfs_instance, base_layer_instance, CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
mobility_hub_instance.load_data(map_area, GEODESIC_CRS)

# 1. Initialize the map canvas with detailed street grids and water bodies
mobility_hub_map1 = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="CartoDB Voyager",  
    zoom_start=13,            
    prefer_canvas=True
)

# 2. Add your underlying demographic walkability/density layer
base_layer_instance.get_folium_plot().add_to(mobility_hub_map1)

# 3. Extract transit data and convert to standard map coordinates
hubs_gdf = mobility_hub_instance._gdf.to_crs(epsg=4326)

# 4. Loop through the hubs and isolate the classifications
for idx, row in hubs_gdf.iterrows():
    tb_val = str(row['trunk_branch_type']).lower().split('.')[-1]
    mode_val = str(row['mode']).lower().split('.')[-1]
    
    # Skip plotting points that are explicitly flagged as NOT being mobility hubs
    if "not_mobility_hub" in tb_val or "not_relevant" in tb_val:
        continue

    # Default styling definitions (Fallback)
    color = "#7f8c8d"  # Grey
    radius = 5
    
    # Fine-grained Mode and Hierarchy Differentiation
    if "cable_car" in mode_val or "cablecar" in mode_val:
        if "trunk" == tb_val:
            color = "#e31700"  # Bright Crimson Red for historic SF Cable Cars
            radius = 8
        elif "branch" == tb_val:
            color = "#ea7164"  # Light Red for historic SF Cable Cars
            radius = 5
    elif "tram" in mode_val or "streetcar" in mode_val or "light_rail" in mode_val:
        if "trunk" == tb_val:
            color = "#00491e"  # Dark Green for Trams / Muni Metro Streetcars
            radius = 8
        elif "branch" == tb_val:
            color = "#2ecc71"  # Green for Trams / Muni Metro Streetcars
            radius = 6
    elif "rail" in mode_val or "subway" in mode_val or "train" in mode_val:
        if "trunk" == tb_val:
            color = "#0dff00"  # Light Lime Green for regional rail (BART / Caltrain)
            radius = 8
        elif "branch" == tb_val:
            color = "#2980b9"  # Deep Blue for Branch Rail lines
            radius = 6
    elif "bus" in mode_val:
        if "trunk" == tb_val:
            color = "#f49eff"  # Light Pink for heavy-frequency Trunk Bus corridors
            radius = 8
        elif "branch" == tb_val:
            color = "#d9d5db"  # Light Gray for Branch Bus lines
            radius = 6
        else:
            color = "#f1c40f"  # Yellow for Local Bus links
            radius = 4

    # --- INDENTED INSIDE THE LOOP: Construct HTML Popup Data Table ---
    od_type = str(row.get('od_type', 'N/A')).split('.')[-1].capitalize()
    od_score = f"{float(row.get('od_score', 0)):.3f}" if row.get('od_score') is not None else "N/A"
    tb_type = str(row.get('trunk_branch_type', 'N/A')).split('.')[-1].capitalize()
    mode_display = str(row.get('mode', 'N/A')).split('.')[-1].capitalize()
    headway = f"{float(row.get('adjusted_headway', 0)):.1f}" if row.get('adjusted_headway') is not None else "N/A"
    freq = f"{float(row.get('total_frequency', 0)):.3f}" if row.get('total_frequency') is not None else "N/A"
    transfer = str(row.get('transfer', 'False')).lower()

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; line-height: 1.4; min-width: 220px;">
        <table style="width: 100%; border-collapse: collapse;">
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Origin / Local?</td><td style="padding: 4px; text-align: right;">{od_type}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Origin / Local Score</td><td style="padding: 4px; text-align: right;">{od_score}</td></tr>
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Trunk / Branch?</td><td style="padding: 4px; text-align: right; font-weight: bold; color: {'#d32f2f' if 'Trunk' in tb_type else '#e67e22'};">{tb_type}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Mode</td><td style="padding: 4px; text-align: right;">{mode_display}</td></tr>
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Minimum Headway (one direction)</td><td style="padding: 4px; text-align: right;">{headway}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Total Frequency (all directions)</td><td style="padding: 4px; text-align: right;">{freq}</td></tr>
            <tr style="background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Transfer</td><td style="padding: 4px; text-align: right; color: {'green' if transfer == 'true' else 'red'};">{transfer}</td></tr>
        </table>
    </div>
    """

    # 5. Draw solid, high-contrast markers onto the canvas (Passing the custom HTML)
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=radius,
        color="#ffffff",       
        weight=1.2,            
        fill=True,
        fill_color=color,      
        fill_opacity=0.9,      
        popup=folium.Popup(popup_html, max_width=350) # <-- FIXED: Uses our rich custom HTML block now!
    ).add_to(mobility_hub_map1)


# 6. Save the output and show map (original finish, completely outside the loop)
mobility_hub_map1.save("mobility_hub_map1.html")
mobility_hub_map1

#### All Elements

In [30]:
# Load data objects
for name, object in all_objects.items():
    print(name)
    if not object.get_is_loaded():
        if name in objects_must_await:
            await object.load_data(map_area, GEODESIC_CRS)
        else:
            object.load_data(map_area, GEODESIC_CRS)

GTFS
BASE
BIKE
CITYBIKES
AFDC
OSM BIKE PARKING


In [31]:
# Main Map
display_map1 = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=12,
    prefer_canvas=True
)
for object in objects_load_order:
    print(object)
    assert all_objects[object].get_is_loaded()
    if len(all_objects[object].gdf) > 0: #TODO: quick fix, add "has_entries" method to each data object instead
        all_objects[object].get_folium_plot().add_to(display_map1)
#all_objects["GTFS"].get_folium_plot().add_to(display_map1)
display_map1
display_map1.save("display_map1.html")

BASE
BIKE
GTFS
AFDC
OSM BIKE PARKING
CITYBIKES


### Example Export Function

In [ ]:
import pathlib
def export_data_object_as_layer(data_object_instance, output_folder):
    """Export data_object_instance to output_folder as a geojson"""
    gdf = data_object_instance.gdf
    gdf.to_file(pathlib.Path(output_folder) / f"{data_object_instance.name}.geojson")

objects_to_export = [gtfs_instance, mobility_hub_instance, citybikes_instance, base_layer_instance, afdc_instance, fta_instance]
for object in objects_to_export:
    print(object.name)
    export_data_object_as_layer(object, "test")
